# 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1.1 Import Libraries

In [ ]:
!pip install category_encoders catboost

In [ ]:
import pickle
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from category_encoders import TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

## 1.2 Load Data

In [ ]:
# Load classification split data
with open("/content/drive/MyDrive/Thesis/model_splits_classification_final.pkl", "rb") as f:
    split_data = pickle.load(f)

# Load cleaned dataset with SES
df_clean_ses = pd.read_pickle("/content/drive/MyDrive/Thesis/df_ses_fix.pkl")

In [ ]:
features_no_ses = split_data["features_no_ses"]
features_with_ses = split_data["features_with_ses"]
categorical_features = split_data["categorical_features"]
numeric_features_with_ses = split_data["numeric_features_with_ses"]

X_train_no_ses = split_data["X_train_no_ses"]
X_test_no_ses = split_data["X_test_no_ses"]
y_train_no_ses = split_data["y_train_no_ses"]
y_test_no_ses = split_data["y_test_no_ses"]

X_train_ses = split_data["X_train_ses"]
X_test_ses = split_data["X_test_ses"]
y_train_ses = split_data["y_train_ses"]
y_test_ses = split_data["y_test_ses"]

In [ ]:
print(X_train_no_ses.shape)
print(X_test_no_ses.shape)
print(X_train_ses.shape)
print(X_test_ses.shape)

print(X_train_no_ses.index.equals(X_train_ses.index))
print(X_test_no_ses.index.equals(X_test_ses.index))

# 2. Preprocessing Setup


In [ ]:
# Check cardinality of categorical features
cardinality_summary = []

for col in categorical_features:
    if col in X_train_no_ses.columns:
        cardinality_summary.append({
            "feature": col,
            "n_unique_train": X_train_no_ses[col].nunique(dropna=False),
            "n_unique_test": X_test_no_ses[col].nunique(dropna=False)
        })

cardinality_df = pd.DataFrame(cardinality_summary).sort_values(
    by="n_unique_train",
    ascending=False
)

cardinality_df

In [ ]:

high_cardinality_features = ["Animal_Breed_clean"]

low_cardinality_features = [
    col for col in categorical_features
    if col not in high_cardinality_features
]

In [ ]:
# No SES
high_cardinality_features_no_ses = [
    col for col in high_cardinality_features
    if col in X_train_no_ses.columns
]

low_cardinality_features_no_ses = [
    col for col in low_cardinality_features
    if col in X_train_no_ses.columns
]

# With SES
high_cardinality_features_ses = [
    col for col in high_cardinality_features
    if col in X_train_ses.columns
]

low_cardinality_features_ses = [
    col for col in low_cardinality_features
    if col in X_train_ses.columns
]

numeric_features_ses = [
    col for col in numeric_features_with_ses
    if col in X_train_ses.columns
]

In [ ]:
print("High-cardinality no SES:", high_cardinality_features_no_ses)
print("Low-cardinality no SES:", low_cardinality_features_no_ses)

print("High-cardinality SES:", high_cardinality_features_ses)
print("Low-cardinality SES:", low_cardinality_features_ses)
print("Numeric SES:", numeric_features_ses)

## 2.1 Check

### 2.1.1 Data Leakage Check

In [ ]:
# Check that target variables are not included in feature matrices
for target_col in ["long_stay", "LOS_days", "log_LOS_days"]:
    for name, X in {
        "X_train_no_ses": X_train_no_ses,
        "X_test_no_ses": X_test_no_ses,
        "X_train_ses": X_train_ses,
        "X_test_ses": X_test_ses
    }.items():
        if target_col in X.columns:
            print(f"WARNING: {target_col} found in {name}")

### 2.1.2 Missing Value Check before modeling

In [ ]:
missing_summary = pd.DataFrame({
    "X_train_no_ses": X_train_no_ses.isna().sum(),
    "X_test_no_ses": X_test_no_ses.isna().sum(),
    "X_train_ses": X_train_ses.isna().sum(),
    "X_test_ses": X_test_ses.isna().sum()
}).fillna(0).astype(int)

missing_summary

In [ ]:
missing_summary.to_csv("/content/drive/MyDrive/Thesis/preprocessing_missing_summary_final.csv")

### 2.1.3 Class balance check

In [ ]:
class_balance = pd.DataFrame({
    "train_no_ses": y_train_no_ses.value_counts(normalize=True),
    "test_no_ses": y_test_no_ses.value_counts(normalize=True),
    "train_ses": y_train_ses.value_counts(normalize=True),
    "test_ses": y_test_ses.value_counts(normalize=True)
})

class_balance

In [ ]:
class_balance.to_csv("/content/drive/MyDrive/Thesis/class_balance_summary.csv")

### 2.1.4 Feature Set Summary

In [ ]:
feature_set_summary = pd.DataFrame({
    "feature_set": ["No SES", "With SES"],
    "n_features_before_encoding": [
        X_train_no_ses.shape[1],
        X_train_ses.shape[1]
    ],
    "n_train": [
        X_train_no_ses.shape[0],
        X_train_ses.shape[0]
    ],
    "n_test": [
        X_test_no_ses.shape[0],
        X_test_ses.shape[0]
    ]
})

feature_set_summary

In [ ]:
feature_set_summary.to_csv(
    "/content/drive/MyDrive/Thesis/feature_set_summary_final.csv",
    index=False
)

### 2.1.5 Cardinality Summary



In [ ]:
cardinality_df.to_csv(
    "/content/drive/MyDrive/Thesis/categorical_cardinality_summary.csv",
    index=False
)

In [ ]:

numeric_transformer_scaled = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

numeric_transformer_unscaled = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

In [ ]:
def impute_numeric_for_catboost(X_train, X_test, numeric_cols):
    X_train_imp = X_train.copy()
    X_test_imp = X_test.copy()

    for col in numeric_cols:
        if col in X_train_imp.columns:
            median_value = X_train_imp[col].median()
            X_train_imp[col] = X_train_imp[col].fillna(median_value)
            X_test_imp[col] = X_test_imp[col].fillna(median_value)

    return X_train_imp, X_test_imp

### 2.3 Logistic Regression

- Sensitive, needs scaling

In [ ]:
logreg_preprocessor_no_ses = ColumnTransformer(
    transformers=[
        ("target_breed", TargetEncoder(smoothing=10), high_cardinality_features_no_ses),
        ("onehot_cat", OneHotEncoder(handle_unknown="ignore"), low_cardinality_features_no_ses)
    ],
    remainder="drop"
)

logreg_preprocessor_ses = ColumnTransformer(
    transformers=[
        ("target_breed", TargetEncoder(smoothing=10), high_cardinality_features_ses),
        ("onehot_cat", OneHotEncoder(handle_unknown="ignore"), low_cardinality_features_ses),
        ("num", numeric_transformer_scaled, numeric_features_ses)
    ],
    remainder="drop"
)

### 2.3.2 Random Forest

In [ ]:
rf_clf_preprocessor_no_ses = ColumnTransformer(
    transformers=[
        ("target_breed", TargetEncoder(smoothing=10), high_cardinality_features_no_ses),
        ("onehot_cat", OneHotEncoder(handle_unknown="ignore"), low_cardinality_features_no_ses)
    ],
    remainder="drop"
)

rf_clf_preprocessor_ses = ColumnTransformer(
    transformers=[
        ("target_breed", TargetEncoder(smoothing=10), high_cardinality_features_ses),
        ("onehot_cat", OneHotEncoder(handle_unknown="ignore"), low_cardinality_features_ses),
        ("num", numeric_transformer_unscaled, numeric_features_ses)
    ],
    remainder="drop"
)

### 2.3.3 CatBoost

In [ ]:
def prepare_catboost_data(X_train, X_test, categorical_cols):
    X_train_cb = X_train.copy()
    X_test_cb = X_test.copy()

    cat_cols_existing = [
        col for col in categorical_cols
        if col in X_train_cb.columns
    ]

    for col in cat_cols_existing:
        X_train_cb[col] = X_train_cb[col].fillna("Unknown").astype(str)
        X_test_cb[col] = X_test_cb[col].fillna("Unknown").astype(str)

    cat_features_idx = [
        X_train_cb.columns.get_loc(col)
        for col in cat_cols_existing
    ]

    return X_train_cb, X_test_cb, cat_cols_existing, cat_features_idx

In [ ]:
# Separate modeling data for catboost
X_train_cb_clf_no_ses, X_test_cb_clf_no_ses, cat_cols_clf_no_ses, cat_features_clf_no_ses = prepare_catboost_data(
    X_train_no_ses,
    X_test_no_ses,
    categorical_features
)

X_train_cb_clf_ses, X_test_cb_clf_ses, cat_cols_clf_ses, cat_features_clf_ses = prepare_catboost_data(
    X_train_ses,
    X_test_ses,
    categorical_features
)

In [ ]:
# Check

print("CatBoost no SES categorical columns:", cat_cols_clf_no_ses)
print("CatBoost no SES categorical indices:", cat_features_clf_no_ses)

print("CatBoost SES categorical columns:", cat_cols_clf_ses)
print("CatBoost SES categorical indices:", cat_features_clf_ses)

In [ ]:
X_train_cb_clf_ses, X_test_cb_clf_ses = impute_numeric_for_catboost(
    X_train_cb_clf_ses,
    X_test_cb_clf_ses,
    numeric_features_ses
)

# 3. Final Check and Save Data

In [ ]:
print(X_train_cb_clf_ses[numeric_features_ses].isna().sum())
print(X_test_cb_clf_ses[numeric_features_ses].isna().sum())

In [ ]:
print(categorical_features)
print("Number of categorical features:", len(categorical_features))

In [ ]:
expected_categorical_features = [
    "Animal_Type",
    "Animal_Breed_clean",
    "Intake_Type",
    "Intake_Subtype_fe",
    "Intake_Condition_clean",  # or Intake_Condition_fe
    "Chip_Status_fe",
    "Animal_Origin_fe",
    "Intake_Year",
    "Intake_Month"
]

print("categorical_features:", categorical_features)
print("Number:", len(categorical_features))
print("Animal_Size included?:", "Animal_Size" in categorical_features)

print("Missing:", set(expected_categorical_features) - set(categorical_features))
print("Extra:", set(categorical_features) - set(expected_categorical_features))

In [ ]:
print("features_no_ses:", features_no_ses)
print("Number of no-SES features:", len(features_no_ses))

print("\nfeatures_with_ses:", features_with_ses)
print("Number of with-SES features:", len(features_with_ses))

In [ ]:
save_path = "/content/drive/MyDrive/Thesis/model_splits_preprocessed.pkl"

split_data = {
    # Feature lists
    "features_no_ses": features_no_ses,
    "features_with_ses": features_with_ses,
    "categorical_features": categorical_features,
    "numeric_features_ses": numeric_features_ses,
    "high_cardinality_features_no_ses": high_cardinality_features_no_ses,
    "high_cardinality_features_ses": high_cardinality_features_ses,
    "low_cardinality_features_no_ses": low_cardinality_features_no_ses,
    "low_cardinality_features_ses": low_cardinality_features_ses,

    # Standard sklearn modeling data
    "X_train_no_ses": X_train_no_ses,
    "X_test_no_ses": X_test_no_ses,
    "y_train_no_ses": y_train_no_ses,
    "y_test_no_ses": y_test_no_ses,

    "X_train_ses": X_train_ses,
    "X_test_ses": X_test_ses,
    "y_train_ses": y_train_ses,
    "y_test_ses": y_test_ses,

    # CatBoost modeling data
    "X_train_cb_clf_no_ses": X_train_cb_clf_no_ses,
    "X_test_cb_clf_no_ses": X_test_cb_clf_no_ses,
    "cat_cols_clf_no_ses": cat_cols_clf_no_ses,
    "cat_features_clf_no_ses": cat_features_clf_no_ses,

    "X_train_cb_clf_ses": X_train_cb_clf_ses,
    "X_test_cb_clf_ses": X_test_cb_clf_ses,
    "cat_cols_clf_ses": cat_cols_clf_ses,
    "cat_features_clf_ses": cat_features_clf_ses,
}

with open(save_path, "wb") as f:
    pickle.dump(split_data, f)

print(f"Saved classification split data to: {save_path}")

In [ ]:
with open(save_path, "rb") as f:
    loaded_split_data = pickle.load(f)

print(loaded_split_data.keys())

print("X_train_no_ses:", loaded_split_data["X_train_no_ses"].shape)
print("X_test_no_ses:", loaded_split_data["X_test_no_ses"].shape)
print("X_train_ses:", loaded_split_data["X_train_ses"].shape)
print("X_test_ses:", loaded_split_data["X_test_ses"].shape)

print("X_train_cb_clf_ses:", loaded_split_data["X_train_cb_clf_ses"].shape)
print("X_test_cb_clf_ses:", loaded_split_data["X_test_cb_clf_ses"].shape)

In [ ]:
print(X_train_cb_clf_ses[numeric_features_ses].isna().sum())
print(X_test_cb_clf_ses[numeric_features_ses].isna().sum())

# Archived Regression

In [ ]:
# # Make Regression data

# # Stage 2 regression subset: only long-stay cases from training/test sets

# X_train_reg_no_ses = X_train_no_ses[y_train_no_ses== 1].copy()
# X_test_reg_no_ses = X_test_no_ses[y_test_no_ses == 1].copy()

# X_train_reg_ses = X_train_ses[y_train_ses == 1].copy()
# X_test_reg_ses = X_test_ses[y_test_ses == 1].copy()

In [ ]:
# # Get LOS_days target using index
# y_train_reg_no_ses = df_clean_ses.loc[X_train_reg_no_ses.index, "LOS_days"]
# y_test_reg_no_ses = df_clean_ses.loc[X_test_reg_no_ses.index, "LOS_days"]

# y_train_reg_ses = df_clean_ses.loc[X_train_reg_ses.index, "LOS_days"]
# y_test_reg_ses = df_clean_ses.loc[X_test_reg_ses.index, "LOS_days"]

In [ ]:
# print("Stage 2 no SES:", X_train_reg_no_ses.shape, X_test_reg_no_ses.shape)
# print("Stage 2 SES:", X_train_reg_ses.shape, X_test_reg_ses.shape)

# print("y train reg no SES:", y_train_reg_no_ses.shape)
# print("y test reg no SES:", y_test_reg_no_ses.shape)
# print("y train reg SES:", y_train_reg_ses.shape)
# print("y test reg SES:", y_test_reg_ses.shape)

### 2.4.1 Ridge Regressor

- linear model -> needs scaling

In [ ]:
# ridge_preprocessor_no_ses = ColumnTransformer(
#     transformers=[
#         ("target_breed", TargetEncoder(smoothing=10), high_cardinality_features_no_ses),
#         ("onehot_cat", OneHotEncoder(handle_unknown="ignore"), low_cardinality_features_no_ses)
#     ],
#     remainder="drop"
# )

# ridge_preprocessor_ses = ColumnTransformer(
#     transformers=[
#         ("target_breed", TargetEncoder(smoothing=10), high_cardinality_features_ses),
#         ("onehot_cat", OneHotEncoder(handle_unknown="ignore"), low_cardinality_features_ses),
#         ("num", numeric_transformer_scaled, numeric_features_ses)
#     ],
#     remainder="drop"
# )

### 2.4.2 Random Forest Regressor

In [ ]:
# rf_reg_preprocessor_no_ses = ColumnTransformer(
#     transformers=[
#         ("target_breed", TargetEncoder(smoothing=10), high_cardinality_features_no_ses),
#         ("onehot_cat", OneHotEncoder(handle_unknown="ignore"), low_cardinality_features_no_ses)
#     ],
#     remainder="drop"
# )

# rf_reg_preprocessor_ses = ColumnTransformer(
#     transformers=[
#         ("target_breed", TargetEncoder(smoothing=10), high_cardinality_features_ses),
#         ("onehot_cat", OneHotEncoder(handle_unknown="ignore"), low_cardinality_features_ses),
#         ("num", numeric_transformer_passthrough, numeric_features_ses)
#     ],
#     remainder="drop"
# )

### 2.4.3 CatBoost Regressor

In [ ]:
# X_train_cb_reg_no_ses, X_test_cb_reg_no_ses, cat_cols_reg_no_ses, cat_features_reg_no_ses = prepare_catboost_data(
#     X_train_reg_no_ses,
#     X_test_reg_no_ses,
#     categorical_features
# )

# X_train_cb_reg_ses, X_test_cb_reg_ses, cat_cols_reg_ses, cat_features_reg_ses = prepare_catboost_data(
#     X_train_reg_ses,
#     X_test_reg_ses,
#     categorical_features
# )

In [ ]:
# print("CatBoost regression no SES categorical columns:", cat_cols_reg_no_ses)
# print("CatBoost regression no SES categorical indices:", cat_features_reg_no_ses)

# print("CatBoost regression SES categorical columns:", cat_cols_reg_ses)
# print("CatBoost regression SES categorical indices:", cat_features_reg_ses)

In [ ]:
# X_train_cb_reg_ses, X_test_cb_reg_ses = impute_numeric_for_catboost(
#     X_train_cb_reg_ses,
#     X_test_cb_reg_ses,
#     numeric_features_ses
# )

In [ ]:
# preprocessing_objects = {
#     # Shared feature info
#     "categorical_features": categorical_features,
#     "numeric_features_with_ses": numeric_features_with_ses,
#     "high_cardinality_features": high_cardinality_features,
#     "low_cardinality_features": low_cardinality_features,

#     "high_cardinality_features_no_ses": high_cardinality_features_no_ses,
#     "low_cardinality_features_no_ses": low_cardinality_features_no_ses,
#     "high_cardinality_features_ses": high_cardinality_features_ses,
#     "low_cardinality_features_ses": low_cardinality_features_ses,
#     "numeric_features_ses": numeric_features_ses,

#     # Stage 1 classification preprocessors
#     "logreg_preprocessor_no_ses": logreg_preprocessor_no_ses,
#     "logreg_preprocessor_ses": logreg_preprocessor_ses,
#     "rf_clf_preprocessor_no_ses": rf_clf_preprocessor_no_ses,
#     "rf_clf_preprocessor_ses": rf_clf_preprocessor_ses,

#     # Stage 1 CatBoost data
#     "X_train_cb_clf_no_ses": X_train_cb_clf_no_ses,
#     "X_test_cb_clf_no_ses": X_test_cb_clf_no_ses,
#     "cat_features_clf_no_ses": cat_features_clf_no_ses,

#     "X_train_cb_clf_ses": X_train_cb_clf_ses,
#     "X_test_cb_clf_ses": X_test_cb_clf_ses,
#     "cat_features_clf_ses": cat_features_clf_ses,

#     # Stage 2 regression data
#     "X_train_reg_no_ses": X_train_reg_no_ses,
#     "X_test_reg_no_ses": X_test_reg_no_ses,
#     "y_train_reg_no_ses": y_train_reg_no_ses,
#     "y_test_reg_no_ses": y_test_reg_no_ses,

#     "X_train_reg_ses": X_train_reg_ses,
#     "X_test_reg_ses": X_test_reg_ses,
#     "y_train_reg_ses": y_train_reg_ses,
#     "y_test_reg_ses": y_test_reg_ses,

#     # Stage 2 regression preprocessors
#     "ridge_preprocessor_no_ses": ridge_preprocessor_no_ses,
#     "ridge_preprocessor_ses": ridge_preprocessor_ses,
#     "rf_reg_preprocessor_no_ses": rf_reg_preprocessor_no_ses,
#     "rf_reg_preprocessor_ses": rf_reg_preprocessor_ses,

#     # Stage 2 CatBoost data
#     "X_train_cb_reg_no_ses": X_train_cb_reg_no_ses,
#     "X_test_cb_reg_no_ses": X_test_cb_reg_no_ses,
#     "cat_features_reg_no_ses": cat_features_reg_no_ses,

#     "X_train_cb_reg_ses": X_train_cb_reg_ses,
#     "X_test_cb_reg_ses": X_test_cb_reg_ses,
#     "cat_features_reg_ses": cat_features_reg_ses,
# }

In [ ]:
# with open("/content/drive/MyDrive/Thesis/preprocessing_stage1_stage2.pkl", "wb") as f:
#     pickle.dump(preprocessing_objects, f)

# print("Preprocessing objects saved successfully.")

In [ ]:
# with open("/content/drive/MyDrive/Thesis/preprocessing_stage1_stage2.pkl", "rb") as f:
#     test_load = pickle.load(f)

# print(test_load.keys())

In [ ]:
# print("CatBoost classification SES missing:")
# print(X_train_cb_clf_ses[numeric_features_ses].isna().sum())
# print(X_test_cb_clf_ses[numeric_features_ses].isna().sum())

# print("CatBoost regression SES missing:")
# print(X_train_cb_reg_ses[numeric_features_ses].isna().sum())
# print(X_test_cb_reg_ses[numeric_features_ses].isna().sum())